# Searching AGOL

## Overview

This notebook will look for any items (web maps, apps) and provide information on each item.

In brief, this notebook will:
1. Iterate over all web content in your org by [items/item type](https://developers.arcgis.com/rest/users-groups-and-items/items-and-item-types.htm)
    1. Iterate over all web maps in your org
    2. Iterate over all feature layers in your org
    3. Iterate over all web apps in your org
    4. Iterate over other items in org

2. Create a list of service ItemIDs
3. Create a list of service URLs
    

## Setup

#### Run this cell to connect to your GIS and get started:

In [1]:
from arcgis.gis import GIS
import pandas as pd
import os

gis = GIS("home")

## The Search

### Web Maps
First, we'll be getting a list of **all** the web maps in your org.

For each web map, it's as simple as using `get_data()` to return the JSON definition of the map as a string. We can then use `find` together with our service URL to see if it shows up.

Our `map_list` is created using **list comprehension** together with a **ternary operator**, which are very handy features for keeping your Python concise.

#### Searching outside your org
Add the parameter `outside_org=True` to the **search** to look for maps from *any* AGOL organization that are shared publicly. Obviously, that's going to be a LOT of information to look through, so only do that if you're certain you want to, then go get some coffee or something while it executes.

In [2]:
webmaps = gis.content.search('', item_type='Web Map', max_items=-1, outside_org=False)
#webmaps

In [3]:
len(webmaps)

290

In [4]:
map_list = [m for m in webmaps]
map_list

[<Item title:"Beta 2 - Commerce City Capital Improvement Projects" type:Web Map owner:C3_GIS_Admin>,
 <Item title:"Special Event Manager" type:Web Map owner:jmoening_c3>,
 <Item title:"Encampment Basemap" type:Web Map owner:rwoolf_c3>,
 <Item title:"Field Map with Parcels" type:Web Map owner:jmoening_c3>,
 <Item title:"Parks Data" type:Web Map owner:jallen_c3>,
 <Item title:"High Injury Network Locator" type:Web Map owner:jmoening_c3>,
 <Item title:"Situation Base Map" type:Web Map owner:C3_GIS_Admin>,
 <Item title:"NS_Reports_D12" type:Web Map owner:jallen_c3>,
 <Item title:"projection issues" type:Web Map owner:jallen_c3>,
 <Item title:"Trails" type:Web Map owner:jallen_c3>,
 <Item title:"Irrigation Points" type:Web Map owner:jallen_c3>,
 <Item title:"test" type:Web Map owner:C3_GIS_Admin>,
 <Item title:"Zoning" type:Web Map owner:aperalta_c3>,
 <Item title:"Streets Repair & Maintenance_workers" type:Web Map owner:aperalta_c3>,
 <Item title:"CDOT Highways" type:Web Map owner:C3_GIS_A

In [5]:
dependencies = pd.concat(
    [
        pd.DataFrame([{'title':m.title, 'id':m.id, 'type':m.type, 'url':f'{gis.url}/home/item.html?id={m.id}', 'owner':m.owner} for m in map_list])
    ]
)

dependencies.reset_index(inplace=True, drop=True)

dependencies.head()

,title,id,type,url,owner
0,Beta 2 - Commerce City Capital Improvement Pro...,79049db6dc6f43a2b8cf5f8b5266795c,Web Map,https://www.arcgis.com//home/item.html?id=7904...,C3_GIS_Admin
1,Special Event Manager,00fd8a29b9f94246b7901614cc731272,Web Map,https://www.arcgis.com//home/item.html?id=00fd...,jmoening_c3
2,Encampment Basemap,eee33cae6df7482498dbff14182a5302,Web Map,https://www.arcgis.com//home/item.html?id=eee3...,rwoolf_c3
3,Field Map with Parcels,dea32144c7144c0ebc19c95f4d22d3b2,Web Map,https://www.arcgis.com//home/item.html?id=dea3...,jmoening_c3
4,Parks Data,7dc3be295d274f878c26be9dc97f68d7,Web Map,https://www.arcgis.com//home/item.html?id=7dc3...,jallen_c3


### Feature Layers
Get a list of **all** the feature layers in your org.

In [6]:
lyr_list = gis.content.search('', item_type='Feature Layer', max_items=-1, outside_org=False)
lyr_list

[<Item title:"SpecialEventAssets" type:Feature Layer Collection owner:jmoening_c3>,
 <Item title:"Encampment Details Field Map App" type:Feature Layer Collection owner:rwoolf_c3>,
 <Item title:"2640" type:Feature Layer Collection owner:dmartinelli_c3>,
 <Item title:"test_AOIs" type:Feature Layer Collection owner:aolson_c3>,
 <Item title:"Commerce City Parcels" type:Feature Layer Collection owner:mcampe_c3>,
 <Item title:"c3_History" type:Feature Layer Collection owner:jallen_c3>,
 <Item title:"CC Encampment Location" type:Feature Layer Collection owner:rwoolf_c3>,
 <Item title:"Police Patrol Beat" type:Feature Layer Collection owner:mcampe_c3>,
 <Item title:"CommunityAddresses" type:Feature Layer Collection owner:C3_GIS_Admin>,
 <Item title:"Northfield and Quebec Layers" type:Feature Layer Collection owner:wnafziger_c3>,
 <Item title:"7255 Monaco Street Layers" type:Feature Layer Collection owner:wnafziger_c3>,
 <Item title:"LiqLicVendors" type:Feature Layer Collection owner:C3_GIS_Adm

### Web Apps
Now we'll be getting a list of **all** the web apps in your org. There are different `item_type`s for each, so we need to perform multiple queries and merge them together. Again, list comprehension makes this a nice short line, rather than nested for-loops.

In [7]:
apptypes = ['Application', 'Dashboard', 'Story Map', 'Web Experience']

app_list = [item for sublist in [gis.content.search('', item_type=t, max_items=-1, outside_org=False) for t in apptypes] for item in sublist]

In [8]:
app_list

[<Item title:"BA Preferences" type:Web Mapping Application owner:wnafziger_c3>,
 <Item title:"BA Web App Void Analysis Data" type:Web Mapping Application owner:wnafziger_c3>,
 <Item title:"City of Commerce City" type:Native Application owner:mcampe_c3>,
 <Item title:"Web App Builder" type:Web Mapping Application owner:bferguson_c3>,
 <Item title:"Parcel Finder Web Map" type:Web Mapping Application owner:C3_GIS_Admin>,
 <Item title:"NewPropertyLookup" type:Desktop Application Template owner:C3_GIS_Admin>,
 <Item title:"Northern Range West of E 470 Map Data" type:Web Mapping Application owner:wnafziger_c3>,
 <Item title:"Commerce City Boundaries Provided by CCC GIS Layers Filter Data" type:Web Mapping Application owner:wnafziger_c3>,
 <Item title:"BA Web App Custom Symbols" type:Web Mapping Application owner:nflock_c3>,
 <Item title:"Commerce City Boundaries Provided by CCC GIS Layers Preferences" type:Web Mapping Application owner:wnafziger_c3>,
 <Item title:"BA Web App Guided Tour Data

### Other Items in Org
Get a list of different types of items in the org site.


In [9]:
item = gis.content.search('', item_type='Map Files', max_items=-1, outside_org=False)
item

[]

In [10]:
item = gis.content.search('', item_type='Tile Layer', max_items=-1, outside_org=False)
item

[]

## Reviewing the Output

Now we've got our lists of maps and apps. But to get them in a user-friendly format, we'll use `pandas` to cobble together a dataframe.

In [29]:
orgData = pd.concat(
    [
        pd.DataFrame([{'title':a.title, 'id':a.id, 'type':a.type, 'url':f'{gis.url}/home/item.html?id={a.id}', 'owner':a.owner} for a in app_list]),
        pd.DataFrame([{'title':m.title, 'id':m.id, 'type':m.type, 'url':f'{gis.url}/home/item.html?id={m.id}', 'owner':m.owner} for m in map_list]),
        pd.DataFrame([{'title':l.title, 'id':l.id, 'type':l.type, 'url':f'{gis.url}/home/item.html?id={l.id}', 'owner':l.owner} for l in lyr_list])
    ]
)

orgData.reset_index(inplace=True, drop=True)

orgData.head()

,title,id,type,url,owner
0,CIP Web App,2fd3018832744b9b910f81aca244da90,Web Mapping Application,https://www.arcgis.com//home/item.html?id=2fd3...,rwoolf_c3
1,Parcel Finder Web Map,9a0bd457466848e18f59fb5538ffc558,Web Mapping Application,https://www.arcgis.com//home/item.html?id=9a0b...,C3_GIS_Admin
2,City of Commerce City,e7db9af47c354712ac622a1cd1446310,Native Application,https://www.arcgis.com//home/item.html?id=e7db...,mcampe_c3
3,Web App Builder C3GIS,a1cc9b850fa249068eb2b7ef7a192ffa,Web Mapping Application,https://www.arcgis.com//home/item.html?id=a1cc...,C3_GIS_Admin
4,Web App Builder,196bd94d9bf0486ebd76aee58ac1b997,Web Mapping Application,https://www.arcgis.com//home/item.html?id=196b...,bferguson_c3


In [34]:
idList = orgData['id']

In [35]:
len(idList)

880

## Export CSV

Now you've got your output, but what do you do with it? Well, you could use `dependencies.to_csv()` to save the dataframe to a file.
If you're just tracking down references to your service to replace or remove them, you can use the `url` column to go straight to the item's page in your portal and make the necessary adjustments.

In [30]:
csv = r'C:\Users\is_olson\Documents\Projects\ArcGIS-for-Server\DependencySearch\test.csv'
myCSV = orgData.to_csv(csv)